# Phase 1: Temporal & Savings Clause Reasoning Engine (v2)
### IPC2BNS-Verify: Date-Conditioned Criminal Law Transition & High Court Split Resolution

This notebook implements and demonstrates the **Temporal Reasoning Engine** for Indian Criminal Law:
1. **Constitutional Mandate**: Article 20(1) (Prohibition of *ex-post facto* penal law).
2. **Statutory Savings Clauses**: Section 531 BNSS 2023, Section 358 BNS 2023, and Section 170 BSA 2023.
3. **High Court Split Resolution**: Resolving jurisdictional divergence (e.g., Kerala HC vs. P&H HC vs. Bombay HC on pending appeals/bail petitions post-July 1, 2024).

In [ ]:
# Step 0: Google Colab & Google Drive Setup
import os, sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
    print("[Colab] Detected Google Colab environment. Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    
    drive_candidates = [
        Path('/content/drive/MyDrive/NLP-rspaper'),
        Path('/content/drive/MyDrive/NLP_rs'),
        Path('/content/drive/MyDrive/research paper/NLP_rs'),
        Path.cwd()
    ]
    
    for cand in drive_candidates:
        if (cand / 'code' / 'src').exists():
            os.chdir(cand)
            print(f"[Colab] Working directory set to: {os.getcwd()}")
            break
            
    if Path("requirements.txt").exists():
        get_ipython().system("pip install -q -r requirements.txt")
except ImportError:
    IN_COLAB = False
    print("[Local] Running in local environment:", os.getcwd())

# Ensure code root is on sys.path
root_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'code']
for p in root_candidates:
    if (p / 'code' / 'src').exists():
        sys.path.insert(0, str(p / 'code'))
        break
    elif (p / 'src').exists():
        sys.path.insert(0, str(p))
        break

print("[Setup] Python path ready:", sys.path[0])

## 1. Import Core Temporal Engine Modules

In [ ]:
from src.temporal.timeline_parser import TimelineParser
from src.temporal.savings_clause_engine import SavingsClauseEngine
from src.temporal.hc_split_resolver import HighCourtSplitResolver

parser = TimelineParser()
engine = SavingsClauseEngine()
split_resolver = HighCourtSplitResolver()
print("Loaded TimelineParser, SavingsClauseEngine, and HighCourtSplitResolver successfully!")

## 2. Test Case 1: Pure Legacy Incident & Procedure (Pre-July 2024)
**Factual Matrix**: Incident occurred in Jan 2024, FIR lodged in Jan 2024.
- Substantive Law: **IPC 1860** (Article 20(1))
- Procedural Law: **CrPC 1973** (Section 531(2)(a) BNSS pending proceeding)

In [ ]:
q1 = "The incident occurred on 10 January 2024. FIR was registered on 15 January 2024 under Section 420 for cheating."
t1 = parser.parse(q1)
r1 = engine.resolve_timeline(t1)

print("Query:", q1)
print("Incident Date :", t1.incident_date)
print("FIR Date      :", t1.fir_date)
print("Substantive   :", r1.substantive_code)
print("Procedural    :", r1.procedural_code)
print("Citations     :", r1.statutory_citations)
print("Rationale     :", r1.reasoning)

## 3. Test Case 2: Transitional Delayed FIR (Pre-July Incident, Post-July FIR)
**Factual Matrix**: Incident occurred on 10 June 2024, but FIR lodged on 15 July 2024.
- Substantive Law: **IPC 1860** (Article 20(1) constitutional bar against ex-post facto penal law)
- Procedural Law: **BNSS 2023** (No pending proceeding on 01-07-2024, so §531(2)(a) savings do not apply)

In [ ]:
q2 = "The alleged assault took place on 10 June 2024. Victim lodged the FIR on 15 July 2024. Which penal section and procedure apply?"
t2 = parser.parse(q2)
r2 = engine.resolve_timeline(t2)

print("Query:", q2)
print("Substantive   :", r2.substantive_code, "(Art 20(1) Non-Retroactivity)")
print("Procedural    :", r2.procedural_code, "(BNSS 2023 - No Pending FIR before July 1)")
print("Citations     :", r2.statutory_citations)

## 4. Test Case 3: High Court Split on Post-July Appeal for Pre-July Conviction
**Factual Matrix**: Trial conviction in May 2024, criminal appeal filed in August 2024.
- In **Kerala & Delhi**: Governed by **BNSS 2023** (*Abdul Khader* / *Prince*)
- In **Punjab & Haryana and Bombay**: Governed by **CrPC 1973** (*Mandeep Singh* / *Digambar*)

In [ ]:
q3 = "Trial convicted appellant under IPC in May 2024. Filing criminal appeal in August 2024."
t3 = parser.parse(q3)
r3 = engine.resolve_timeline(t3)

print("Is Contested High Court Split:", r3.is_contested_split)
print("Procedural Resolution        :", r3.procedural_code)
if r3.split_details:
    print("\n--- Split Issue ---")
    print(r3.split_details.get("issue"))
    print("\n--- Jurisdictional Precedents ---")
    for k, v in r3.split_details.get("jurisdictions", {}).items():
        print(f"  * {k}: {v['position']} ({v['case']})")
    print("\n--- Legal Practice Advisory ---")
    print(r3.split_details.get("advisory"))

## 5. Verify Full Temporal Test Suite

In [ ]:
!pytest code/tests/test_temporal.py -v